# 134 — Sensores, actuadores y fusión

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("robotics", seed=134)
assert result["kind"] == "robotics"
assert result["evidence"]
show(result)


## Solución 1 — Pasos 2 y 3 del filtro

**Paso 2** (`z=2.3`): `x̂⁻ = 1.111+1 = 2.111`; `P⁻ = 0.556+0.25 = 0.806`;
`K = 0.806/1.806 = 0.446`; `x̂ = 2.111 + 0.446·0.189 = 2.195`;
`P = 0.554·0.806 = 0.446`. Coincide con el README.

**Paso 3** (`z=3.1`): `x̂⁻ = 3.195`; `P⁻ = 0.696`; `K = 0.696/1.696 = 0.410`;
`x̂ = 3.195 + 0.410·(3.1−3.195) = 3.156`; `P = 0.590·0.696 = 0.411`.
Observa que K y P se van estabilizando: el filtro entra en régimen
estacionario.


In [ ]:
x, P = 0.0, 1.0
Q, R, u = 0.25, 1.0, 1.0
for z in [1.2, 2.3, 3.1]:
    x_pred, P_pred = x + u, P + Q
    K = P_pred / (P_pred + R)
    x = x_pred + K * (z - x_pred)
    P = (1 - K) * P_pred
    print(f"z={z}: K={K:.3f} x={x:.3f} P={P:.3f}")


## Solución 2 — Fusión estática

```text
x̂ = (σ₂²·z₁ + σ₁²·z₂)/(σ₁²+σ₂²) = (0.16·4.8 + 0.04·5.2)/0.20 = 4.88 m
σ̂² = (0.04·0.16)/0.20 = 0.032
```

La estimación queda cerca del LiDAR (el sensor preciso pesa 4× más). La
varianza fusionada (0.032) es menor que la del mejor sensor (0.04) porque dos
fuentes independientes de información siempre suman evidencia: la precisión
(1/σ²) es aditiva: `1/0.04 + 1/0.16 = 31.25 ⇒ σ̂² = 0.032`.


In [ ]:
z1, v1 = 4.8, 0.04
z2, v2 = 5.2, 0.16
x = (v2*z1 + v1*z2) / (v1+v2)
v = (v1*v2) / (v1+v2)
print(round(x, 3), round(v, 4))


## Solución 3 — Casos límite de Q y R

- (a) Con `Q=0`, P solo decrece en cada corrección y nunca crece: `P→0` y por
  tanto `K→0`. El filtro acaba ignorando el sensor — correcto *solo si* el
  modelo es realmente perfecto.
- (b) Con `R→∞`, `K→0` desde el primer paso: el filtro se convierte en
  **integración pura del modelo** (dead reckoning).
- (c) La odometría pura es el caso (b): sin correcciones, `P = P + Q` crece
  linealmente sin cota — esa es exactamente la deriva que motiva SLAM en la
  clase 135.


In [ ]:
x, P = 0.0, 1.0
Q, u = 0.25, 1.0
R = 1e9  # sensor practicamente desconectado
for t in range(20):
    x, P = x + u, P + Q
    K = P / (P + R)
    P = (1 - K) * P
print(f"tras 20 pasos sin sensor util: K={K:.6f}, P={P:.2f} (crece sin cota)")


## Solución 4 — Contrato del lab

Los elementos de `evidence` que describen observaciones del episodio (lecturas,
decisiones registradas) juegan el papel de mediciones: son los hechos
inspeccionables. `limitations` nunca está vacío por diseño: igual que P
comunica cuánta confianza merece x̂, `limitations` comunica cuánta confianza
merece la demo. Un resultado sin incertidumbre declarada es tan sospechoso
como un filtro con P=0.


In [ ]:
from ai_evolution.labs import run_lab

for seed in (134, 777):
    r = run_lab("robotics", seed=seed)
    assert r["kind"] == "robotics"
    assert r["limitations"], "un lab honesto declara limitaciones"
    print(seed, "->", len(r["evidence"]), "evidencias")
